In [2]:
import os
import sys

import torch
import numpy as np
import pandas as pd

import shared.utils as su
from utils.video import read_frames_decord
from notebooks.eval_care_retrieval import (
    load_data,
    compute_metrics,
    print_metrics_as_latex_row,
)
import json

In [4]:
from models.internvl3 import AutoEncoder

# model_path = "/work/piyush/pretrained_checkpoints/InternVL3-8B"
model_path = "/work/piyush/experiments/CaRe/InternVL3-8B/covr/chiral10k-covr10k/merged_checkpoint/"

encoder = AutoEncoder.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2",
)

frames = read_frames_decord(video_path='../assets/demo.mp4', num_frames=16)
text = "This video features a man slicing tomatoes in the kitchen."
vision_emb = encoder.encode_vision(frames.unsqueeze(0))
text_emb = encoder.encode_text(text)
print(f'Vision embedding shape: {vision_emb.shape}')
print(f'Text embedding shape: {text_emb.shape}')
# print(f'Cosine similarity: {cosine_similarity(vision_emb, text_emb)}')

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


Loading EncoderForInternVL3 from /work/piyush/experiments/CaRe/InternVL3-8B/covr/chiral10k-covr10k/merged_checkpoint/


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Vision embedding shape: torch.Size([1, 3584])
Text embedding shape: torch.Size([1, 3584])


In [9]:
# Load data
dataset = "epic"
df = load_data(dataset=dataset)
df = df.drop_duplicates(subset=['id', 'text_id']).reset_index(drop=True)
df.shape

Number of rows:  3109
Sample row: 
{
    "narration_id": "P01_11_1",
    "participant_id": "P01",
    "video_id": "P01_11",
    "narration_timestamp": "00:00:01.700",
    "start_timestamp": "00:00:01.56",
    "stop_timestamp": "00:00:02.45",
    "start_frame": 93,
    "stop_frame": 147,
    "narration": "put down plate",
    "verb": "put-down",
    "verb_class": 1,
    "noun": "na",
    "noun_class": 2,
    "all_nouns": "['plate']",
    "all_noun_classes": "[2]",
    "start_sec": 1.56,
    "stop_sec": 2.45,
    "id": "P01/videos/P01_11_1.6_2.4",
    "chiral_label": 0.0,
    "chiral_triplet_id": "ecff8160",
    "template": "put-down plate",
    "noun_value": "plate",
    "text_id": "ecff8160_0.0",
    "video_path": "/scratch/shared/beegfs/piyush/datasets/EPIC-Kitchens-100/cut_clips/P01/videos/P01_11_1.6_2.4.MP4"
}


(3108, 24)

In [10]:
# Compute text features
text_ids = df['text_id'].unique()
texts_feat = {}
j = 0
for text_id in su.log.tqdm_iterator(text_ids, desc='Computing text features'):
    text = df[df.text_id == text_id].template.unique()[0]
    zt = encoder.encode_text(text).squeeze(0)
    zt = torch.nn.functional.normalize(zt, dim=-1)
    texts_feat[text_id] = zt.cpu().float()
    if j == 0:
        print("Text embedding: ", zt.shape)
    j += 1
len(texts_feat)

Computing text features:   0%|          | 0/132 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Text embedding:  torch.Size([3584])


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for

132

In [ ]:
# Compute video features
video_paths = df.video_path.unique()
video_ids = df.id.unique()
video_feat = {}
j = 0
for video_path in su.log.tqdm_iterator(video_paths, desc='Computing video features'):
    frames = read_frames_decord(video_path=video_path, num_frames=16).unsqueeze(0)
    zv = encoder.encode_vision(frames)[0]
    zv = torch.nn.functional.normalize(zv, dim=-1)
    video_feat[video_ids[j]] = zv.cpu().float()
    if j == 0:
        print("Video embedding: ", zv.shape)
    j += 1


Computing video features:   0%|          | 0/3108 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Video embedding:  torch.Size([3584])


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for

In [ ]:
metrics = compute_metrics(df, video_feat, texts_feat, show_metrics=False)
print_metrics_as_latex_row(metrics, sep='& ')

# Save metrics
save_metrics = True
if save_metrics:
    save_dir = os.path.join(model_path, 'metrics')
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, f'metrics-{dataset}.json'), 'w') as f:
        json.dump(metrics, f)
else:
    print(f"Metrics not saved.")
    print(json.dumps(metrics, indent=4))


save_embs = True
if save_embs:
    save_dir = os.path.join(model_path, 'embs')
    os.makedirs(save_dir, exist_ok=True)
    print(f"Saving embeddings to {save_dir}")
    torch.save(video_feat, os.path.join(save_dir, f'video_feat-{dataset}.pt'))
    torch.save(texts_feat, os.path.join(save_dir, f'texts_feat-{dataset}.pt'))